### Creazione e addestramento dei modelli

In [ ]:
import pandas as pd

def load_and_split_data(train_path, test_path, target_col='target_assignments'):
    """
    Carica i file CSV e separa le feature (X) dal target (y).
    """
    # Caricamento dei dataset
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    
    # Rimozione di eventuali colonne di timestamp non necessarie per il training
    # (es. se presenti nel CSV ma non utili al modello)
    if 'planning_date' in df_train.columns:
        df_train = df_train.drop(columns=['planning_date'])
    if 'planning_date' in df_test.columns:
        df_test = df_test.drop(columns=['planning_date'])

    # Separazione X e y
    X_train = df_train.drop(columns=[target_col])
    y_train = df_train[target_col]
    
    X_test = df_test.drop(columns=[target_col])
    y_test = df_test[target_col]
    
    print(f"Dataset caricati correttamente:")
    print(f"- X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"- X_test:  {X_test.shape}, y_test:  {y_test.shape}")
    
    return X_train, y_train, X_test, y_test

# --- Esecuzione ---
# Sostituisci i nomi dei file con i tuoi percorsi reali
X_train, y_train, X_test, y_test = load_and_split_data('data/train_dataset.csv', 'data/test_dataset.csv')

In [ ]:
import xgboost as xgb
import numpy as np
import pandas as pd

def train_xgboost_quantiles(X_train, y_train, X_test):
    """
    Addestra 3 modelli XGBoost per stimare il 10°, 50° (mediana) e 90° percentile.
    Richiede xgboost >= 2.0.0.
    """
    print("Addestramento modello limite inferiore (10° percentile)...")
    xgb_q10 = xgb.XGBRegressor(
        objective='reg:quantileerror', 
        quantile_alpha=0.10,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        random_state=42
    )
    xgb_q10.fit(X_train, y_train)

    print("Addestramento modello mediano (50° percentile - Predizione principale)...")
    xgb_q50 = xgb.XGBRegressor(
        objective='reg:quantileerror', 
        quantile_alpha=0.50,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        random_state=42
    )
    xgb_q50.fit(X_train, y_train)

    print("Addestramento modello limite superiore (90° percentile)...")
    xgb_q90 = xgb.XGBRegressor(
        objective='reg:quantileerror', 
        quantile_alpha=0.90,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        random_state=42
    )
    xgb_q90.fit(X_train, y_train)
    
    # Generazione delle predizioni sul Test Set
    pred_q10 = xgb_q10.predict(X_test)
    pred_q50 = xgb_q50.predict(X_test)
    pred_q90 = xgb_q90.predict(X_test)
    
    return pred_q10, pred_q50, pred_q90

# Esecuzione
q10, q50, q90 = train_xgboost_quantiles(X_train, y_train, X_test)

In [ ]:
def prepare_asp_facts(X_test, y_test, q10, q50, q90):
    """
    Combina le predizioni con i dati di contesto e calcola lo score di incertezza.
    """
    results_df = X_test.copy()
    
    # La predizione da suggerire all'ASP arrotondata all'intero più vicino
    results_df['predicted_assignments'] = np.round(q50).astype(int)
    
    # Valore reale (per eventuale validazione, non da passare all'ASP)
    results_df['actual_assignments'] = y_test.values
    
    # Calcolo dell'Incertezza (Prediction Interval Width)
    # Più il valore è alto, maggiore è l'incertezza del modello.
    results_df['uncertainty_score'] = q90 - q10
    
    # Estrazione del burdenScore (che è già presente in X_test in base alla tua EDA)
    # Assicuriamoci che si chiami correttamente
    burden_col = [col for col in X_test.columns if 'burdenScore' in col][0]
    
    print("\nAnteprima dei dati pronti per essere convertiti in fatti ASP:")
    display_cols = ['predicted_assignments', 'uncertainty_score', burden_col, 'density_ratio']
    display(results_df[display_cols].head())
    
    return results_df

results_df = prepare_asp_facts(X_test, y_test, q10, q50, q90)